# Compression de lames SVS

Benchmark des méthodes de compression sur un fichier SVS complet.

**Important** : l'original est déjà compressé en JPEG (~Q70-80). Les compressions sans perte (Deflate, LZW...) seront TOUJOURS plus grosses car elles décompressent puis recompressent sans perte.

On ne teste donc que les compressions **avec perte** qui peuvent réellement réduire la taille.

In [1]:
from pathlib import Path
import time
import shutil
import openslide

try:
    import pyvips
    print("pyvips OK")
except Exception as e:
    print("ERREUR pyvips :", e)

DATA_DIR = Path("/Users/nath/Desktop/stageHCL/data_stage")
OUT_DIR = Path("/Users/nath/Desktop/stageHCL/tests_compression")
OUT_DIR.mkdir(exist_ok=True)

SVS_PATH = DATA_DIR / "TCGA-3M-AB46-01Z-00-DX1.70F638A0-BDCB-4BDE-BBFE-6D78A1A08C5B.svs"
orig_size = SVS_PATH.stat().st_size
print(f"Lame : {SVS_PATH.name}")
print(f"Taille originale : {orig_size/(1024*1024):.1f} Mo")
print(f"Espace libre : {shutil.disk_usage(OUT_DIR).free/(1024*1024*1024):.1f} Go")

pyvips OK
Lame : TCGA-3M-AB46-01Z-00-DX1.70F638A0-BDCB-4BDE-BBFE-6D78A1A08C5B.svs
Taille originale : 1133.7 Mo
Espace libre : 46.2 Go


objc[39945]: Class GNotificationCenterDelegate is implemented in both /Users/nath/Desktop/stageHCL/.venv/lib/python3.14/site-packages/openslide_bin/libopenslide.1.dylib (0x110b211a0) and /opt/homebrew/Cellar/glib/2.88.0/lib/libgio-2.0.0.dylib (0x1100e44c8). This may cause spurious casting failures and mysterious crashes. One of the duplicates must be removed or renamed.


In [2]:
# Affiche les infos de la lame
slide = openslide.OpenSlide(str(SVS_PATH))
for k, v in slide.properties.items():
    if 'compress' in k.lower() or 'quality' in k.lower() or 'jpeg' in k.lower() or 'tile' in k.lower():
        print(f"{k}: {v}")
print(f"\nNiveaux : {slide.level_count}")
print(f"Dimensions niveau 0 : {slide.level_dimensions[0]}")
slide.close()

openslide.level[0].tile-height: 240
openslide.level[0].tile-width: 240
openslide.level[1].tile-height: 240
openslide.level[1].tile-width: 240
openslide.level[2].tile-height: 240
openslide.level[2].tile-width: 240
openslide.level[3].tile-height: 240
openslide.level[3].tile-width: 240

Niveaux : 4
Dimensions niveau 0 : (105575, 87609)


In [3]:
def compresser(suffix: str, compression: str, **kwargs):
    """Compresse la lame entière en TIFF pyramidal."""
    out_path = OUT_DIR / f"{SVS_PATH.stem}_{suffix}.tiff"
    img = pyvips.Image.new_from_file(str(SVS_PATH))
    t0 = time.perf_counter()
    img.tiffsave(
        str(out_path),
        tile=True, tile_width=256, tile_height=256,
        pyramid=True,
        compression=compression,
        **kwargs
    )
    dt = time.perf_counter() - t0
    size = out_path.stat().st_size
    ratio = orig_size / size
    print(f"Fichier : {out_path.name}")
    print(f"Taille  : {size/(1024*1024):.1f} Mo")
    print(f"Ratio   : {ratio:.2f}x")
    print(f"Temps   : {dt:.1f} s")
    return out_path

## Test 1 : JPEG Q=50
Compression JPEG agressive. Perte de qualité visible probable.

In [4]:
f = compresser("jpeg_50", "jpeg", Q=50)

Fichier : TCGA-3M-AB46-01Z-00-DX1.70F638A0-BDCB-4BDE-BBFE-6D78A1A08C5B_jpeg_50.tiff
Taille  : 696.8 Mo
Ratio   : 1.63x
Temps   : 52.0 s


## Test 2 : JPEG Q=75
Compression JPEG modérée. Bon compromis taille/qualité probable.

In [5]:
f = compresser("jpeg_75", "jpeg", Q=75)

Fichier : TCGA-3M-AB46-01Z-00-DX1.70F638A0-BDCB-4BDE-BBFE-6D78A1A08C5B_jpeg_75.tiff
Taille  : 922.6 Mo
Ratio   : 1.23x
Temps   : 51.2 s


## Test 3 : JPEG Q=90

Etant donnée que le fichier apres 'compression' est plus lourd, cela veut dire qu'il est deja compresser avec perte Q<90

In [6]:
f = compresser("jpeg_90", "jpeg", Q=90)

Fichier : TCGA-3M-AB46-01Z-00-DX1.70F638A0-BDCB-4BDE-BBFE-6D78A1A08C5B_jpeg_90.tiff
Taille  : 3303.4 Mo
Ratio   : 0.34x
Temps   : 53.1 s


## Test 4 : JPEG 2000
Transformée en ondelettes, standard médical. Attention : lent et fichier volumineux car encodé sans perte par défaut.

In [8]:
f = compresser("jp2k", "jp2k")

Error: unable to call tiffsave
  vips2tiff: Maximum TIFF file size exceeded
wbuffer_write: write failed
system error: Unknown error: -1
vips2tiff: Maximum TIFF file size exceeded


## Récapitulatif

Exécute cette cellule après chaque test pour voir le tableau se remplir.

In [9]:
import pandas as pd

rows = []
for f in sorted(OUT_DIR.glob("*.tiff")):
    rows.append({
        "methode": f.stem.split("_")[-1],
        "taille_Mo": round(f.stat().st_size / (1024*1024), 1),
        "ratio": round(orig_size / f.stat().st_size, 2),
    })

rows.insert(0, {"methode": "ORIGINAL (SVS)", "taille_Mo": round(orig_size/(1024*1024),1), "ratio": 1.0})
df = pd.DataFrame(rows)
display(df)

,methode,taille_Mo,ratio
0,ORIGINAL (SVS),1133.7,1.00
1,jp2k,4096.0,0.28
2,50,696.8,1.63
3,75,922.6,1.23
4,90,3303.4,0.34


## Nettoyage

Supprime les fichiers de test un par un pour libérer de l'espace.

In [10]:
# Lister les fichiers de test
for f in sorted(OUT_DIR.glob("*.tiff")):
    print(f"{f.name}  ({f.stat().st_size/(1024*1024):.1f} Mo)")

TCGA-3M-AB46-01Z-00-DX1.70F638A0-BDCB-4BDE-BBFE-6D78A1A08C5B_jp2k.tiff  (4096.0 Mo)
TCGA-3M-AB46-01Z-00-DX1.70F638A0-BDCB-4BDE-BBFE-6D78A1A08C5B_jpeg_50.tiff  (696.8 Mo)
TCGA-3M-AB46-01Z-00-DX1.70F638A0-BDCB-4BDE-BBFE-6D78A1A08C5B_jpeg_75.tiff  (922.6 Mo)
TCGA-3M-AB46-01Z-00-DX1.70F638A0-BDCB-4BDE-BBFE-6D78A1A08C5B_jpeg_90.tiff  (3303.4 Mo)
